# Data Management and Training Logistics with PyTorch

Here, we introduce to you how you can do everything we saw in the second session with PyTorch.

## The Dataset

The `Dataset` class is the "wrapper" for your raw data. It decouples the data storage from the model logic. To create a custom dataset, you must implement three methods:

- `__init__`: Either loads the data or defines the file paths to load it.
- `__len__`: Returns the total number of samples.
- `__getitem__`: Retrieves the i-th sample, applying the specificated transformations.

This class allows us to "lazy load" our data; if all of the data needed for training does not fit in RAM at once, we load it batch by batch.

Here's a couple of examples:

```python
import pandas as pd
from torch.utils.data import Dataset

class TabularDataset(Dataset):
  """
  Loads a CSV with numeric features and a target column.

  Assumes the file has a header row, all columns are numeric, and the last column is the target
  """
  def __init__(self, csv_path: str, target_col: str = "price"):
    dataframe = pd.read_csv(csv_path)

    feature_cols = [col for col in df.columns if col != target_col]
    self.X = torch.tensor(df[feature_cols].values, dtype=torch.float32)
    self.y = torch.tensor(df[target_col].values, dtype=torch.float32).unsqueeze(1)

  def __len__(self):
    return len(self.X)
  
  def __getitem__(self, int: idx):
    return self.X[idx], self.y[idx]

```

```python
class ImageFolderDataset(Dataset):
    """
    Reads images from a flat folder and labels from a CSV.
    
    Expected CSV layout (data/labels.csv):
        filename,label
        img001.jpg,0
        img002.jpg,1
        ...
    
    Expected image directory:
        data/images/img001.jpg
        data/images/img002.jpg
        ...
    """

    def __init__(self, images_dir: str, labels_csv: str, transform=None):
        self.images_dir = Path(images_dir)
        self.transform  = transform
        
        df = pd.read_csv(labels_csv)
        self.filenames = df["filename"].tolist()
        self.labels    = df["label"].tolist()

    def __len__(self):
        return len(self.filenames)

    def __getitem__(self, idx):
        img_path = self.images_dir / self.filenames[idx]
        image    = Image.open(img_path).convert("RGB")
        
        if self.transform:
            image = self.transform(image)
        
        label = torch.tensor(self.labels[idx], dtype=torch.long)
        return image, label
```

## The DataLoader

The `DataLoader` wraps a `Dataset` instance and takes care of the logistics:
- **Batching**: grouping samples into mini-batches
- **Shuffling**: randomizing the order of the samples each epoch.
- **Parallel loading**: using multiple CPU workers to pre-fetch data while the GPU trains.

You can define how it is going to handle all of these aspects when initializing an instance of it. Considering the previous `TabularDataset` example:

```python
from torch.utils.data import DataLoader

tabular_dataset = TabularDataset(csv_path = "path/to/csv", target_col = "area")
tabular_dataloader = DataLoader(tabular_dataset, batch_size=32, shuffle=True, num_workers=8)
```

## Optimizers

The available optimizers can be be imported using `torch.optim`:

```python
import torch.optim as optim

# SGD with momentum: the classic baseline
sgd_optimizer = optim.SGD(
    model.parameters(),
    lr=0.01,
    momentum=0.9,       # exponential moving average of past gradients
    weight_decay=1e-4,  # L2 regularization (penalizes large weights)
)

# Adam: adaptive learning rates per parameter — usually the best default
adam_optimizer = optim.Adam(
    model.parameters(),
    lr=1e-3,
    betas=(0.9, 0.999),  # decay rates for first and second moment estimates
    weight_decay=1e-5,
)

# AdamW: Adam with decoupled weight decay — preferred for transformers
adamw_optimizer = optim.AdamW(
    model.parameters(),
    lr=1e-3,
    weight_decay=1e-2,
)
```


The first parameter they receive when initializing them are the model's parameters (you will have to instantiate it first), and then the parameters related to the optimizer: learning rate and the rest of it depending on the selected optimizer. [Here](https://docs.pytorch.org/docs/stable/optim.html)'s the documentation for them.

## The Training Loop

If we put it all together, the training loop looks like this:

```python
def train_epoch(model, loader, optimizer, criterion):
  model.train()
  total_loss = 0.0

  for X_batch, y_batch in loader:
    X_batch = X_batch.to(device)
    y_batch = y_batch.to(device)

    optimizer.zero_grad()

    predictions = model(X_batch)

    loss = criterion(predictions, y_batch)

    loss.backward()

    optimizer.step()

    total_loss += loss.item()

  return total_loss / len(loader)
```

## Exercises

### The Dataset class

Here, we give you a simple dataset class that contains a list of numbers and their squares.

In [27]:
from torch.utils.data import Dataset
import torch

class SquaresDataset(Dataset):
  def __init__(self, size):
    data = [(i, i**2) for i in range(size)]
    self.X = torch.tensor([x for x, _ in data], dtype=torch.float32)
    self.y = torch.tensor([y for _, y in data], dtype=torch.float32)

  def __len__(self):
    return len(self.X)

  def __getitem__(self, idx):
    return self.X[idx], self.y[idx]

dataset = SquaresDataset(10)

1. Print how many samples it has.

2. Access the 3rd sample and print the input (value) and the label (squared value)

3. Modify the dataset to add a small amount of random noise to each `y` value. Do the previous exercise to see how the `y` value changes.

4. Write a minimal custom `Dataset` class from scratch. Nothing too complicated. Suggestion: create a `Dataset` that holds a list of student scores (input) and a pass/fail label (if score is higher than 5, 1; otherwise 0). Remember the three methods you have to implement.

### The DataLoader

Use the Dataset you created previously (or the one we provided) for the next exercises.

1. Create a `DataLoader` with `batch_size=4` and `shuffle=False`.

2. Loop over the DataLoader and print each batch (inputs and labels).

3. Repeat with `shuffle=True` and compare the order.

4. Try `batch_size=1` and `batch_size=len(dataset)` &mdash; what do you notice?

### Optimizers

In [32]:
def train_epoch(model, dataloader, optimizer, criterion, device):
  model.train()
  total_loss = 0.0

  for X_batch, y_batch in dataloader:
    X_batch = X_batch.unsqueeze(1).to(device)
    y_batch = y_batch.unsqueeze(1).to(device)

    optimizer.zero_grad()

    predictions = model(X_batch)

    loss = criterion(predictions, y_batch)

    loss.backward()

    optimizer.step()

    total_loss += loss.item()

  return total_loss / len(dataloader)


In [33]:
import torch.nn as nn

class SimpleMLP(nn.Module):
  def __init__(self, input_size, hidden_size, output_size):
    super().__init__()
    self.fc1 = nn.Linear(input_size, hidden_size)
    self.fc2 = nn.Linear(hidden_size, output_size)
    self.relu = nn.ReLU()

  def forward(self, x):
    x = self.fc1(x)
    x = self.relu(x)
    x = self.fc2(x)
    return x

In [34]:
from torch.utils.data import DataLoader
from torch import optim
import torch

model = SimpleMLP(1, 10, 1)
dataloader = DataLoader(dataset, batch_size=4, shuffle=True)
optimizer = optim.SGD(model.parameters(), lr=0.01)
criterion = nn.MSELoss()
device = "cuda" if torch.cuda.is_available() else "cpu"

model.to(device)

SimpleMLP(
  (fc1): Linear(in_features=1, out_features=10, bias=True)
  (fc2): Linear(in_features=10, out_features=1, bias=True)
  (relu): ReLU()
)

1. As the code is now, you are using SGD as the optimizer. Run a training during 10 epochs and print the loss at each epoch.

In [ ]:
# for epoch in range(10): ...

2. Run it again changing the optimizer to Adam (same learning rate) and compare.

3. Try a very high learning rate (e.g. 10) with SGD &mdash; what happens?

4. Plot the loss curves side by side with matplotlib. Check the documentation [here](https://matplotlib.org/stable/users/explain/quick_start.html).